In [ ]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR

from LeakDetector import LIKELY_HASHES, ENCODINGS_NO_ROT, LeakDetector

sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False, use_cache=True)


In [4]:
import warnings

from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))


### Load the SST hostnames to use in leak detection

In [5]:
df_sst_hosts = pd.read_csv('detected_sst_req_hostnames.csv')
sst_req_hostnames = set(df_sst_hosts.req_hostname.unique())

In [7]:
def search_prod_name(search_term, site_domain, exp_mode, consent_mode, df, sst_req_hostnames):
    # Initialize a leak detector with the product name as the search term
    print(f"Searching: {search_term} in {site_domain} | {exp_mode} - {consent_mode}")

    # Initialize LeakDetector instance
    ld = LeakDetector(
        [search_term],
        encoding_set=ENCODINGS_NO_ROT,
        hash_set=None,
        encoding_layers=2,
        hash_layers=0,
        debugging=False
    )

    # Filter DataFrame for the specific site_domain and conditions
    df_filtered = df[(df.site_domain == site_domain) &
                     (df.exp_mode == exp_mode) &
                     (df.consent_mode == consent_mode)]
    assert len(df_filtered) > 0, f"No data found for {site_domain} {exp_mode} - {consent_mode}"

    # Detect leaks in URLs and POST data
    df_filtered['url_leaks'] = df_filtered['url'].apply(ld.check_url, encoding_layers=2)
    df_filtered['post_leaks'] = df_filtered['post_data'].apply(ld.check_post_data, encoding_layers=2)

    # Helper function to create a leak record
    def create_leak_record(row, leak, leak_type, is_sst):
        return {
            'site_domain': row['site_domain'],
            'search_term': search_term,
            'leak_type': leak_type,
            'leak_encoding': leak[0] if len(leak) > 1 else '',
            'leak_value': leak[1] if len(leak) > 1 else leak[0],
            'consent_mode': consent_mode,
            'exp_mode': exp_mode,
            'country': row['country'],
            'req_hostname': row['req_hostname'],
            'is_tracker': row['is_tracker'],
            'is_cname_tracker': row['is_cname_tracker'],
            'sst': is_sst,
            'entity': row['entity'],
            'url': row['url'],
            'post_data': row['post_data'],
        }

    # Collect individual leaks
    leak_records = []
    for idx, row in df_filtered.iterrows():
        is_sst = row['req_hostname'] in sst_req_hostnames

        # Collect URL leaks
        leak_records.extend(create_leak_record(row, url_leak, 'url_leak', is_sst) for url_leak in row['url_leaks'])

        # Collect POST data leaks
        leak_records.extend(create_leak_record(row, post_leak, 'post_leak', is_sst) for post_leak in row['post_leaks'])

    return pd.DataFrame(leak_records)


In [5]:
def search_prod_name(seach_term, site_domain, exp_mode, consent_mode, df, sst_req_hostnames):
    # Initialize a leak detector with the product name as the search term
    print(f"Searching: {seach_term} in {site_domain} | {exp_mode} - {consent_mode}")

    # Initialize leak detector
    ld = LeakDetector(
        [seach_term],
        encoding_set=ENCODINGS_NO_ROT,
        hash_set=None,
        encoding_layers=2,
        hash_layers=0,
        debugging=False
    )

    # Filter DataFrame for the specific site_domain
    df_filtered = df[(df.site_domain == site_domain) &
                     (df.exp_mode == exp_mode) &
                     (df.consent_mode == consent_mode)]
    assert len(df_filtered) > 0, f"No data found for {site_domain} {exp_mode} - {consent_mode}"
    # Apply leak detection to URLs and post_data
    df_filtered['url_leaks'] = df_filtered['url'].apply(lambda url: tuple(ld.check_url(url, encoding_layers=2)))
    df_filtered['post_leaks'] = df_filtered['post_data'].apply(lambda post_data: tuple(ld.check_post_data(post_data, encoding_layers=2)))

    # Initialize list to store individual leak details
    leak_records = []

    # Iterate through the filtered DataFrame and collect individual leaks
    for idx, row in df_filtered.iterrows():
        is_sst = row['req_hostname'] in sst_req_hostnames
        # Collect each URL leak
        for url_leak in row['url_leaks']:
            leak_records.append({
                'site_domain': row['site_domain'],
                'search_term': seach_term,
                'leak_type': 'url_leak',
                'leak_encoding': url_leak[0] if len(url_leak) > 1 else '',
                'leak_value': url_leak[1] if len(url_leak) > 1 else url_leak[0],
                # add all the other columns
                'consent_mode': consent_mode,
                'exp_mode': exp_mode,
                'country': row['country'],
                # df_filtered.columns
                'req_hostname': row['req_hostname'],
                'is_tracker': row['is_tracker'],
                'is_cname_tracker': row['is_cname_tracker'],
                'sst': is_sst,
                'entity': row['entity'],
                'url': row['url'],
                'post_data': row['post_data'],
            })

        # Collect each POST data leak
        for post_leak in row['post_leaks']:
            leak_records.append({
                'site_domain':  row['site_domain'],
                'search_term': seach_term,
                'leak_type': 'post_leak',
                'leak_encoding': post_leak[0] if len(post_leak) > 1 else '',
                'leak_value': post_leak[1] if len(post_leak) > 1 else post_leak[0],
                'consent_mode': consent_mode,
                'exp_mode': exp_mode,
                'country': row['country'],
                # df_filtered.columns
                'req_hostname': row['req_hostname'],
                'is_tracker': row['is_tracker'],
                'is_cname_tracker': row['is_cname_tracker'],
                'sst': is_sst,
                'entity': row['entity'],
                'url': row['url'],
                'post_data': row['post_data'],
            })

    return pd.DataFrame(leak_records)



In [ ]:

from tld import get_fld
from utils import CONSENT_MODES, MEASUREMENT_MODES

PRODUCT_NAMES_CSV = "product_names.csv"

dfl = pd.DataFrame(columns=[
    'site_domain', 'search_term', 'leak_type', 'leak_encoding',
    'leak_value', 'consent_mode', 'exp_mode', 'country', 'req_hostname',
    'is_tracker', 'is_cname_tracker', 'sst', 'entity', 'url', 'post_data'])

# read the product names from the CSV
prod_names = pd.read_csv(PRODUCT_NAMES_CSV)

# we only check the "tracking" hars
cnt = 0
# for exp_mode in MEASUREMENT_MODES:
for __, row in prod_names.iterrows():
    site_domain = get_fld(row['site'], fix_protocol=True)
    leak_df = search_prod_name(row['product_name'], site_domain, row['exp_mode'], row['consent_mode'], df, sst_req_hostnames)
    dfl = pd.concat([dfl, leak_df], ignore_index=True)
    cnt += 1
    if cnt > 100:
        print(f"Processed {cnt} product names")
        # break

dfl['req_domain'] = dfl['req_hostname'].apply(lambda x: get_fld(x, fix_protocol=True))
dfl.head()


In [ ]:
dfl['third_party'] = dfl['req_domain'] != dfl['site_domain']
dfl.third_party.value_counts().to_frame()

In [ ]:
dfl.sst.value_counts().to_frame()

In [ ]:
dfl[dfl.sst].req_hostname.nunique()

### Only take 3rd P or SST requests

In [13]:
dfl = dfl[dfl.third_party | dfl.sst]
dfla = dfl[dfl.consent_mode=='Accept']
dflr = dfl[dfl.consent_mode=='Reject']

## Top domains receiving product leaks

In [ ]:

tmp_df = dfl.pivot_table(
    index='req_domain',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(10)

In [ ]:
print(tmp_df.sort_values(by='Accept', ascending=False).head(10).to_latex())

## Top entities receiving product leaks

In [ ]:

tmp_df = dfl[dfl.entity!=''].pivot_table(
    index='entity',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(5)

In [ ]:
print(tmp_df.sort_values(by='Accept', ascending=False).head(5).to_latex())

In [ ]:
dfl.groupby('leak_encoding')['site_domain'].nunique().sort_values(ascending=False).to_frame().head(10)

In [ ]:
dfl.is_tracker.value_counts().to_frame()

In [ ]:
dfl.leak_type.value_counts(normalize=True).to_frame()

In [ ]:
dfl[dfl.is_tracker == False].req_hostname.unique()

In [ ]:
dfl[dfl.leak_encoding=='base64']

In [ ]:

tmp_df = dfl.pivot_table(
    index='country',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(10)

In [ ]:
print(tmp_df.sort_values(by='Accept', ascending=False).to_latex())

In [ ]:
dfl.groupby(['country', 'site_domain', 'leak_value', 'consent_mode'])['req_domain'].nunique().sort_values(ascending=False).to_frame().head(10)

In [ ]:
dflr.groupby(['country', 'site_domain', 'leak_value', 'consent_mode'])['req_domain'].nunique().sort_values(ascending=False).to_frame().head(10)

In [ ]:
dflr.groupby(['country', 'site_domain', 'leak_value'])['req_domain'].agg(lambda x: set(x)).sort_values(ascending=False).to_frame().head(10)

In [ ]:
agg_funcs = {
    'req_domain': ['nunique', lambda x: set(x)]
}

result = dflr.groupby(['country', 'site_domain', 'leak_value']).agg(agg_funcs)
result.columns = ['_'.join(col).strip() for col in result.columns.values]
result = result.sort_values(by='req_domain_nunique', ascending=False).head(10)

print(result.to_latex())
# result

In [ ]:
agg_funcs = {
    'entity': ['nunique', lambda x: set(x)]
}

result = dflr.groupby(['country', 'site_domain', 'leak_value']).agg(agg_funcs)
result.columns = ['_'.join(col).strip() for col in result.columns.values]
result = result.sort_values(by='entity_nunique', ascending=False).head(10)

# print(result.to_latex())
result

In [ ]:
dfla[(dfla.site_domain=='farmaciasdirect.es')].drop_duplicates(subset=['leak_value', 'req_domain']).sort_values('entity')[
    ['leak_type', 'leak_value', 'req_hostname', 'entity', 'sst', 'is_tracker']]

In [ ]:
dflr[(dflr.site_domain=='efarma.com')].drop_duplicates(subset=['leak_value', 'leak_type', 'req_domain'])[['site_domain', 'leak_type', 'leak_value', 'req_hostname']]

In [ ]:
dfl.leak_value.value_counts().head(10).to_frame()

In [ ]:
dfl.leak_type.value_counts().head(10).to_frame()

In [ ]:
dfl.groupby(['exp_mode', 'consent_mode'])['site_domain'].\
    nunique().to_frame()


In [ ]:
# top req_hostnames that receive the most data. Sort by distinct sites req_hostname is seen on
# ensure that the count is based on distnict (site_domain)
dfl.drop_duplicates(subset=['site_domain', 'req_hostname']).req_hostname.value_counts().head(10)
# all_leak_df['req_hostname'].value_counts().head(10)/

In [ ]:
dfl.drop_duplicates(['site_domain', 'req_hostname', 'leak_value']).leak_value.value_counts().to_frame()

In [ ]:
dfl.drop_duplicates(subset=['site_domain', 'exp_mode', 'consent_mode', 'req_hostname', 'leak_type', 'leak_value'])

In [ ]:
# count how many websites a particular domain received a leak
dfl.groupby(['site_domain', 'req_hostname']).size().reset_index(name='count').sort_values(by='count', ascending=False)
# df.groupby(['site_domain', 'req_hostname']).size().reset_index(name='count').sort_values(by='count', ascending=False)